# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** rank every page by the percentile of its prior average search position (the average position from March 1-15). A worse (higher) prior position gets a higher priority score. This is the single signal validated in the capstone analysis - a richer blend and three separate machine learning models were all tested against it, and none beat it by a statistically significant margin, so the simplest version is used here.

**Reason codes this rule can output:**
- Visible in search but earning zero clicks in the prior window
- Inconsistent visibility - fewer than half of tracked days had impressions
- Already ranking below the corpus median position
- Ranking position is the primary driver - no secondary flags

In [2]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

print("Connected.")

Connected.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import numpy as np
import os

page_level = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT content_hash_id,
               AVG(gsc_avg_position) as avg_position_h2
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date > '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT f.*, s.avg_position_h2
    FROM first_half f
    JOIN second_half s ON f.content_hash_id = s.content_hash_id
""").df()

print(f"Pages scored: {len(page_level)}")

# Rank-normalized single-signal score: worse prior position -> higher priority.
# Percentile ranking is used instead of min/max scaling, since min/max was found in the
# capstone analysis to be badly distorted by a few extreme outlier values.
sorted_pos = np.sort(page_level["avg_position_h1"].values)
ranks = np.searchsorted(sorted_pos, page_level["avg_position_h1"].values, side="right") / len(sorted_pos)
page_level["baseline_score"] = 1 - ranks

page_level["percentile"] = page_level["baseline_score"].rank(pct=True)
page_level["priority_tier"] = np.select(
    [page_level["percentile"] >= 0.90, page_level["percentile"] >= 0.70],
    ["Refresh Immediately", "Monitor"], default="No Action"
)

median_position = page_level["avg_position_h1"].median()

def build_reasons(row):
    reasons = []
    if row["avg_clicks_h1"] == 0 and row["avg_impressions_h1"] > 0:
        reasons.append("Visible in search but earning zero clicks in the prior window")
    if row["active_days_h1"] < 8 and row["priority_tier"] != "No Action":
        reasons.append("Inconsistent visibility - fewer than half of tracked days had impressions")
    if row["avg_position_h1"] > median_position:
        reasons.append("Already ranking below the corpus median position")
    return " | ".join(reasons) if reasons else "Ranking position is the primary driver - no secondary flags"

page_level["reason_code"] = page_level.apply(build_reasons, axis=1)

ranked = page_level.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

output_cols = ["rank", "content_hash_id", "baseline_score", "percentile", "priority_tier", "reason_code",
               "avg_impressions_h1", "avg_clicks_h1", "avg_position_h1", "active_days_h1"]

os.makedirs("work/outputs", exist_ok=True)
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Written: work/outputs/baseline_action_score.csv ({len(ranked)} rows)")
print(f"\nTier distribution:\n{ranked['priority_tier'].value_counts()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages scored: 141467
Written: work/outputs/baseline_action_score.csv (141467 rows)

Tier distribution:
priority_tier
No Action              99037
Monitor                28283
Refresh Immediately    14147
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Leakage check: confirm no second-half/outcome column was used in the scoring itself
feature_cols_used = ["avg_impressions_h1", "avg_clicks_h1", "avg_position_h1", "active_days_h1"]
assert "avg_position_h2" not in feature_cols_used, "Leak: outcome column present in scoring features"
print("Leakage check passed: no second-half/outcome column used in the score.")

# Weak picks: pages flagged highly on very little data are the ones most likely to be wrong -
# a page seen only once or twice can look identical to a real decline purely by chance.
weak_picks = ranked[(ranked["priority_tier"] == "Refresh Immediately") & (ranked["active_days_h1"] <= 2)]
print(f"\nWeak picks (Refresh Immediately with 2 or fewer active days): {len(weak_picks)}")
weak_picks[output_cols].head(10)

**Which picks look weak, and why:** pages flagged `Refresh Immediately` on very few active days (shown above) are the weakest picks - a page seen only once or twice can look like a decline purely by chance, not a real pattern. Name a specific row from the table above once you see it.

**Leakage confirmed:** no client names, product flags, or second-half/future-window columns (`avg_position_h2` or anything from March 16-31) were used anywhere in the scoring logic - confirmed by the assertion above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.